# GNN-Pruning — full from-scratch reproduction (Colab / A100)

Trains and records **every** result from scratch — 5 methods (dense + magnitude
+ 3 Wanda) × all datasets × architectures (GCN / GAT / GraphSAGE / GPR-GNN) × 9
sparsities, with **3-seed** error bars on the core cells and **minibatch
(neighbour-sampled)** training on the large graphs (Reddit, ogbn-arxiv, Yelp,
ogbn-products, Flickr). This is the artifact a third party runs to replicate the
study.

- It **wipes prior results and retrains everything** — nothing is skipped because
  "we already did it". (The one exception is resuming after a disconnect — see below.)
- The neighbour sampler is **pure PyTorch** — no `pyg-lib`/`torch-sparse` install.
- `reddit/gat` is excluded (full-batch attention eval is infeasible, ~178 GiB).

**Setup:** `Runtime → A100 GPU` (Colab Pro), and enable **background execution**.
**Runtime:** expect **a few hours** (the heterophilic cells are 3 seeds × 10
splits across 5 methods; the large graphs train via minibatch on the A100).
Results persist to your Drive, so a disconnect is recoverable.


## 1. Confirm GPU

In [1]:
import torch
assert torch.cuda.is_available(), "Set Runtime → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0),
      "| VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
      "| torch:", torch.__version__)

GPU: NVIDIA A100-SXM4-80GB | VRAM(GB): 85.1 | torch: 2.11.0+cu128


## 2. Mount Drive (results persist here, so a disconnect is recoverable)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/gnn-pruning'   # a third party points this at their own Drive
os.makedirs(DRIVE_ROOT + '/data', exist_ok=True)
os.makedirs(DRIVE_ROOT + '/results', exist_ok=True)
print('persisting under', DRIVE_ROOT)

Mounted at /content/drive
persisting under /content/drive/MyDrive/gnn-pruning


## 3. Clone `main` + install (no pyg-lib needed)

In [3]:
%cd /content
![ -d GNN-Pruning-Research ] || git clone --branch main https://github.com/Mike-Mans/GNN-Pruning-Research.git
%cd /content/GNN-Pruning-Research
!git fetch origin && git checkout main && git pull --ff-only

/content
Cloning into 'GNN-Pruning-Research'...
remote: Enumerating objects: 2137, done.
remote: Counting objects: 100% (2137/2137), done.
remote: Compressing objects: 100% (684/684), done.
remote: Total 2137 (delta 620), reused 2020 (delta 509), pack-reused 0 (from 0)
Receiving objects: 100% (2137/2137), 4.14 MiB | 34.71 MiB/s, done.
Resolving deltas: 100% (620/620), done.
/content/GNN-Pruning-Research
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.


In [4]:
# Symlink data/ and results/ to Drive (persist across sessions).
import os, shutil
for d in ['data', 'results']:
    if os.path.islink(d):
        continue
    if os.path.exists(d):
        shutil.rmtree(d)
    os.symlink(f'{DRIVE_ROOT}/{d}', d)
!pip -q install torch_geometric ogb rdkit
!pip -q install -e .
print('install done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 46.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gnn-pruning (pyproject.toml) ... done
install done


## 4. ⚠️ Fresh start — wipe all prior results
**Run this once** to start a clean from-scratch run. **Skip it** if you are *resuming* after a disconnect (then steps 1–3 + 5 pick up where it left off).

In [5]:
import glob
n = len(glob.glob(f'{DRIVE_ROOT}/results/**/metrics.json', recursive=True))
!rm -rf {DRIVE_ROOT}/results/*
print(f'wiped {n} prior result cells — starting from scratch')

wiped 0 prior result cells — starting from scratch


## 5. Train the full pipeline — no-pruning first
no-pruning trains every dense baseline; the 4 pruning methods then load those checkpoints. Idempotent, so a disconnected run resumes here.

In [6]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
CONFIGS = {
    'no-pruning':      'src/gnn_pruning/configs/no_pruning.yaml',
    'magnitude':       'src/gnn_pruning/configs/magnitude.yaml',
    'wanda-uniform':   'src/gnn_pruning/configs/wanda_uniform.yaml',
    'wanda-degree':    'src/gnn_pruning/configs/wanda_degree.yaml',
    'wanda-per-class': 'src/gnn_pruning/configs/wanda_per_class.yaml',
}
for method, cfg in CONFIGS.items():
    print(f'\n===== {method} =====', flush=True)
    rc = subprocess.run([sys.executable, '-m', 'gnn_pruning.cli', 'sweep',
                         '--method', method, '--config', cfg],
                        env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f'{method} done (rc={rc})', flush=True)
print('\nFULL SWEEP COMPLETE')


===== no-pruning =====
no-pruning done (rc=0)

===== magnitude =====
magnitude done (rc=0)

===== wanda-uniform =====
wanda-uniform done (rc=0)

===== wanda-degree =====
wanda-degree done (rc=0)

===== wanda-per-class =====
wanda-per-class done (rc=0)

FULL SWEEP COMPLETE


## 6. Sanity check — coverage, big-dataset baselines, failures

In [7]:
import glob, json
print('=== completed cells per method ===')
for m in ['no-pruning','magnitude','wanda-uniform','wanda-degree','wanda-per-class']:
    print(f'  {m:16s}: {len(glob.glob(f"results/{m}/*/*/seed-*/split-*/metrics.json"))} runs')
print('\n=== large-dataset dense baselines (minibatch-trained) ===')
for f in sorted(glob.glob('results/no-pruning/*/*/seed-0/split-0/metrics.json')):
    d = f.split('/')[2]
    if d in {'reddit','ogbn-arxiv','yelp','ogbn-products','flickr'}:
        m = json.load(open(f)); print(f"  {d:14s}/{f.split('/')[3]:10s} {m['metric_value']:.3f}")
print('\n=== failures (expect none) ===')
!grep -h 'FAILED' results/*/run.log | sort | uniq -c | head

=== completed cells per method ===
  no-pruning      : 353 runs
  magnitude       : 353 runs
  wanda-uniform   : 353 runs
  wanda-degree    : 353 runs
  wanda-per-class : 353 runs

=== large-dataset dense baselines (minibatch-trained) ===
  flickr        /graphsage  0.500
  ogbn-arxiv    /graphsage  0.549
  reddit        /gcn        0.356
  reddit        /graphsage  0.535
  yelp          /graphsage  0.290

=== failures (expect none) ===
      1   FAILED in 8.4s (rc=1)
      1   FAILED in 8.5s (rc=1)
      2   FAILED in 8.6s (rc=1)
      1   FAILED in 9.5s (rc=1)


## 7. Regenerate the comprehensive results report

In [8]:
import os
env = {**os.environ, 'GNN_ROOT': os.getcwd()}
import subprocess, sys
subprocess.run([sys.executable, 'scripts/make_results_doc.py'], env=env)
print('wrote results/results_comprehensive.md')

wrote results/results_comprehensive.md


## 8. Download all results (summaries + metrics + plots + report)

In [9]:
import shutil, os, glob
os.makedirs('/content/export/results', exist_ok=True)
files = (glob.glob('results/**/summary.csv', recursive=True)
         + glob.glob('results/**/metrics.json', recursive=True)
         + glob.glob('results/**/run.log', recursive=True)
         + glob.glob('results/**/*.png', recursive=True)
         + glob.glob('results/results_comprehensive.md'))
for f in files:
    dst = '/content/export/' + f
    os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst)
shutil.make_archive('/content/gnn_results', 'zip', '/content/export')
print('archived files:', len(files))
from google.colab import files; files.download('/content/gnn_results.zip')

archived files: 1807


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes
- **Resume after a disconnect:** re-run steps 1–3 and 5 (skip step 4's wipe). Finished
  cells are skipped; the rest continue.
- **`*.pt` checkpoints** are gitignored / not downloaded (large, regenerable).
- **`reddit/gat`** is excluded by design (infeasible full-batch attention).
